In [4]:
import pandas as pd

df = pd.read_csv("/content/ai4i2020.csv")
df.head()

print("Info")
df.info()

Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int

Define a dataframe named df that reads from dataset ai4i2020.csv that contains synthetic dataset used in machine learning for predictive maintenance. All numerical features are stored in appropriate numeric formats. Downloaded from https://www.kaggle.com/datasets/chcarneiro/ai4i2020-csv.

In [8]:
print("NaN Contains")
df.isna().sum()

NaN Contains


,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


Check if the dataset contains any NaN or empty data. No missing values were detected in any feature.

In [6]:
print("Any Duplicate")
df.duplicated().sum()

Any Duplicate


np.int64(0)

Check if the dataset contains any duplicated data. All numerical features are stored in appropriate numeric formats.

In [ ]:
# Define target + features

target = "Machine failure"

# Drop identifiers
drop_cols = ["UDI", "Product ID"]

X = df.drop(columns=drop_cols + [target])
y = df[target].astype(int)

X.head(), y.value_counts()

(  Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
 0    M                298.1                    308.6                    1551   
 1    L                298.2                    308.7                    1408   
 2    L                298.1                    308.5                    1498   
 3    L                298.2                    308.6                    1433   
 4    L                298.2                    308.7                    1408   
 
    Torque [Nm]  Tool wear [min]  TWF  HDF  PWF  OSF  RNF  
 0         42.8                0    0    0    0    0    0  
 1         46.3                3    0    0    0    0    0  
 2         49.4                5    0    0    0    0    0  
 3         39.5                7    0    0    0    0    0  
 4         40.0                9    0    0    0    0    0  ,
 Machine failure
 0    9661
 1     339
 Name: count, dtype: int64)

Column "UDI" dropped because it is not useful and "Product ID" dropped because it potential to leakage (The model may accidentally learn information that it should not logically have access to when making predictions.)

In [ ]:
# Preprocess (encode Type) + train/test split
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

cat_cols = ["Type"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=260226, stratify=y
)

y_train.mean(), y_test.mean()

(np.float64(0.033875), np.float64(0.034))

This steps needs to transform the categorical columns "Type" into numerical using OneHotEncoder because ML models cannot use text directly. Data count used for training is 80% and the rest 20% is used for testing.

In [ ]:
# Build models: Random Forest & LightGBM
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=260226,
    n_jobs=-1,
    class_weight="balanced", # like neg/pos
)

# LightGBM
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos

lgbm = LGBMClassifier(
    n_estimators=500,
    random_state=260226,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    verbosity=-1 # to hide warnings
)

rf_pipe = Pipeline([("prep", preprocess), ("model", rf)])
lgbm_pipe = Pipeline([("prep", preprocess), ("model", lgbm)])

In this project, two tree-based classification models are trained to predict Machine failure: Random Forest and Light Gradient Boosting Machine (LightGBM). Both models output failure probabilities, which can be used as risk scores.

Random Forest:
*   n_estimators=500 means 500 trees are trained to make the model more stable.
*   n_jobs=-1 enables parallel training using all CPU cores, and random_state ensures reproducible results.
*   class_weight="balanced" is used to handle class imbalance by giving higher weight to the failure class so the model does not ignore rare failure cases.

LightGBM:
*   n_estimators=500 to make them identical with Random Forest.
*   n_jobs=-1 enables parallel training using all CPU cores, and random_state ensures reproducible results.
*   scale_pos_weight = (number of non-failure samples) / (number of failure samples) to handle class imbalance.

The preprocessing step applies one hot encoding to the categorical feature Type and passes numerical sensor features unchanged, so same preprocessing is applied consistently during training and testing

In [ ]:
# Evaluate: ROC-AUC + PR-AUC + confusion matrix
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

threshold = 0.5

# Evaluate model based on pipes above
def eval_model(name, pipe):
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)

    roc = roc_auc_score(y_test, proba)
    pr  = average_precision_score(y_test, proba)

    print(f"=== {name} ===")
    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC : {pr:.4f}  (important for imbalanced data)")
    print("Confusion matrix @0.5:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=5))
    return proba

rf_proba  = eval_model("Random Forest", rf_pipe)
lgbm_proba = eval_model("LightGBM", lgbm_pipe)

=== Random Forest ===
ROC-AUC: 0.9814
PR-AUC : 0.9716  (important for imbalanced data)
Confusion matrix @0.5:
 [[1932    0]
 [   2   66]]
              precision    recall  f1-score   support

           0    0.99897   1.00000   0.99948      1932
           1    1.00000   0.97059   0.98507        68

    accuracy                        0.99900      2000
   macro avg    0.99948   0.98529   0.99228      2000
weighted avg    0.99900   0.99900   0.99899      2000

=== LightGBM ===
ROC-AUC: 0.9943
PR-AUC : 0.9757  (important for imbalanced data)
Confusion matrix @0.5:
 [[1932    0]
 [   2   66]]
              precision    recall  f1-score   support

           0    0.99897   1.00000   0.99948      1932
           1    1.00000   0.97059   0.98507        68

    accuracy                        0.99900      2000
   macro avg    0.99948   0.98529   0.99228      2000
weighted avg    0.99900   0.99900   0.99899      2000



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


This steps evaluate the two models's performance using threshold 0.5 using ROC-AUC, PR-AUC, confusion matrix, and classification metrics. The models were first trained on the training dataset and then used to predict failure probabilities on the test dataset. These probabilities were converted into binary predictions using a default threshold. ROC-AUC was used to measure the overall classification capability, while PR-AUC was emphasized due to class imbalance in failure cases. The confusion matrix and classification report were used to analyze precision, recall, and F1-score, providing insight into the model's ability to correctly detect machine failures and avoid false alarms.

In [ ]:
# Risk scoring
risk_df = X_test.copy()
risk_df["y_true"] = y_test.values
risk_df["risk_rf"] = rf_proba
risk_df["risk_lgbm"] = lgbm_proba

risk_df.sort_values("risk_lgbm", ascending=False).head(15) # show top 15 highest-risk machines

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF,y_true,risk_rf,risk_lgbm
3943,L,302.3,311.4,1333,66.7,205,0,0,1,1,0,1,0.984,1.0
4024,M,302.1,311.0,1351,60.3,207,0,0,0,1,0,1,0.794,1.0
4729,L,303.4,311.8,1306,61.0,215,0,1,0,1,0,1,0.988,1.0
160,L,298.4,308.2,1282,60.7,216,0,0,0,1,0,1,0.978,1.0
4976,L,303.7,312.7,1359,56.8,194,0,0,0,1,0,1,0.836,1.0
1583,L,298.2,308.4,1310,61.0,189,0,0,0,1,0,1,0.954,1.0
4406,L,302.5,310.3,1326,58.5,55,0,1,0,0,0,1,0.992,1.0
3684,L,302.0,311.2,1270,65.3,182,0,0,0,1,0,1,0.734,1.0
7167,L,300.3,310.3,1362,60.3,206,0,0,0,1,0,1,0.982,1.0
4383,L,301.7,309.5,1298,65.5,229,0,1,0,1,0,1,0.976,1.0


This steps display risk scoring table based on the test dataset. The true failure label was combined with predicted probabilities from Random Forest and LightGBM. These probabilities represent the estimated risk of machine failure and allow machines to be ranked based on failure likelihood. This step transforms model predictions into actionable risk scores for predictive analytics with the threshold is 0.5 .